In [1]:
import numpy as np
import imageio
import dask
from dask import delayed
import dask.bag as db
import time

In [2]:
def readImg(path):
    img = imageio.imread(path)
    im = np.array(img, dtype='uint8')
    return im

def writeImg(path, buf):
    imageio.imwrite(path, buf)

def part_median_filter(local_data):
    part_id = local_data[0]
    first = local_data[1]
    end = local_data[2]
    buf = local_data[3]

    # Extract the relevant part of the image
    new_buf = buf[int(first):int(end), :, :]

    # Compute median filter
    for i in range(1, new_buf.shape[0] - 1):
        for j in range(1, new_buf.shape[1] - 1):
            pixels_list = [
                new_buf[i-1, j-1], new_buf[i-1, j], new_buf[i-1, j+1],
                new_buf[i, j-1], new_buf[i, j], new_buf[i, j+1],
                new_buf[i+1, j-1], new_buf[i+1, j], new_buf[i+1, j+1]
            ]
            new_buf[i, j] = np.median(pixels_list, axis=0)

    return part_id, new_buf

In [3]:
def main():
    img_path = 'lena_noisy.jpg'
    img_buf = readImg(img_path)
    print('SHAPE', img_buf.shape)
    nx, ny, nz = img_buf.shape

    ###########################################################################
    # SPLIT IMAGES IN NB_PARTITIONS PARTS
    nb_partitions = 8
    print("NB PARTITIONS : ", nb_partitions)
    data = []
    begin = 0
    block_size = nx // nb_partitions
    for ip in range(nb_partitions):
        end = min(begin + block_size, nx)
        data.append([ip, begin, end, img_buf])
        begin = end

    ###########################################################################
    # CREATE DASK BAG
    data_bag = db.from_sequence(data, npartitions=nb_partitions)

    ###########################################################################
    # PARALLEL MEDIAN FILTER COMPUTATION
    start_time = time.time()
    result_bag = data_bag.map(part_median_filter)
    result_data = result_bag.compute()
    end_time = time.time()

    print(f"Dask Execution Time: {end_time - start_time} seconds")

    # Initialize new_img_buf
    new_img_buf = np.zeros((0, ny, nz), dtype='uint8')

    ###########################################################################
    # COMPUTE NEW IMAGE RESULTS FROM RESULT BAG
    for ip in range(nb_partitions):
        new_img_buf = np.concatenate((new_img_buf, result_data[ip][1]), axis=0)

    print('CREATE NEW PICTURE FILE')
    writeImg('lena_filter_DASK.jpg', new_img_buf)

if __name__ == '__main__':
    main()

SHAPE (128, 128, 3)
NB PARTITIONS :  8


/tmp/ipykernel_10737/4148301076.py:2: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path)


Dask Execution Time: 0.38914966583251953 seconds
CREATE NEW PICTURE FILE
